内存存储：ChatMessageHistory 默认是在内存中运行的，重启程序数据就会丢失。如果需要永久存储，请使用 SQLChatMessageHistory 或 RedisChatMessageHistory 等持久化实现。

#### 场景1:记忆存储

In [36]:
from functools import partial

from dateutil.parser import parser
from langchain_classic.chains.conversation.base import ConversationChain
from langchain_classic.chains.llm import LLMChain
from langchain_classic.chains.summarize.refine_prompts import prompt_template
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory
# ChatMessageHistory的实例化
from langchain_community.chat_message_histories import ChatMessageHistory, RedisChatMessageHistory
from openai import conversations

# 初始化一个内存记录器
history = ChatMessageHistory()

# 添加消息
history.add_user_message('你好，我叫曹雪芹')
history.add_user_message('你好，郭德纲有什么可以帮你的')

# 获取记录
print(history.messages)


[HumanMessage(content='你好，我叫曹雪芹', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，郭德纲有什么可以帮你的', additional_kwargs={}, response_metadata={})]


#### 场景2:对接LLM

In [38]:
# 方式一：基础用法
from langchain_openai import ChatOpenAI
import os
import dotenv
from langchain_community.chat_message_histories import ChatMessageHistory

dotenv.load_dotenv(dotenv_path='../.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")

history = ChatMessageHistory()

# history.add_user_message('你好，我叫曹雪芹')
# history.add_user_message('你好，郭德纲有什么可以帮你的')
history.add_user_message('我叫什么名字')

model = ChatOpenAI(
    model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)

res = model.invoke(history.messages)
print(res.content)


我无法查看或获取您的个人信息，包括您的名字。为了保护您的隐私和安全，我不会询问或尝试获取您的个人身份信息。如果您愿意，您可以直接告诉我您的名字，我会记住并使用它来称呼您。


## 自动注入上下文RunnableWithMessageHistory

In [20]:
# 方式二:自动注入上下文
'''
MessagesPlaceholder 是构建“多轮对话模板”的核心，它让你在模板中预留一个插入历史对话的位置。
RunnableWithMessageHistory 负责管理历史记录的存取、注入和更新，让你无需手动拼接历史。
这种模式非常适合构建聊天机器人、客服系统等需要上下文记忆的应用。

在开发阶段，ChatMessageHistory 非常方便。但上线前，请务必替换为 RedisChatMessageHistory 等持久化方案，防止服务重启导致数据丢失。
'''
from langchain_openai import ChatOpenAI
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate
import os
import dotenv
from langchain_core.prompts import MessagesPlaceholder

# RedisChatMessageHistory()

dotenv.load_dotenv(dotenv_path='../.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")

model = ChatOpenAI(
    model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)
# 1. 构建一个带历史占位符的提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一位友好的助手"),
    MessagesPlaceholder(variable_name="history"),  # 历史消息占位符
    ("human", "{input}")  # 当前用户输入
])

# 2. 构建链：prompt → model
chain = prompt | model

# 用于存储不同 session_id 的历史记录 (实际中可以用 Redis/Postgres 替代)
store = {}


def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# 包装你的模型
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# 调用
config = {"configurable": {"session_id": "session1"}}
response = chain_with_history.invoke({"input": "你好，我叫小明"}, config=config)
print('AI回复：', response.content)
response1 = chain_with_history.invoke({"input": "毕业于哈佛学院"}, config=config)
print('AI回复：', response1.content)
# 下次调用时，模型会自动记住“我叫小明”
response2 = chain_with_history.invoke({"input": "小明毕业于什么学校？"}, config=config)
print('AI回复：', response2.content)

print('-----')
print(store["session1"])
# 输出: 你叫小明。

AI回复： 你好，小明！很高兴认识你。有什么我可以帮助你的吗？
AI回复： 哇，那可真是个了不起的成就！哈佛学院的教育质量非常高，你在那里的学习经历一定非常丰富。你在哈佛主修什么专业呢？有没有什么特别难忘的经历或收获？
AI回复： 小明，你刚刚提到自己毕业于哈佛学院，对吗？那是一个非常著名的学府。你在哈佛的学习和生活一定有很多精彩的故事吧？
-----
Human: 你好，我叫小明
AI: 你好，小明！很高兴认识你。有什么我可以帮助你的吗？
Human: 毕业于哈佛学院
AI: 哇，那可真是个了不起的成就！哈佛学院的教育质量非常高，你在那里的学习经历一定非常丰富。你在哈佛主修什么专业呢？有没有什么特别难忘的经历或收获？
Human: 小明毕业于什么学校？
AI: 小明，你刚刚提到自己毕业于哈佛学院，对吗？那是一个非常著名的学府。你在哈佛的学习和生活一定有很多精彩的故事吧？


## ConversationBufferMemory的使用（已经过期，不在维护）

In [9]:
# 举例一
from langchain_classic.memory.buffer import ConversationBufferMemory

memory = ConversationBufferMemory()
# inputs
memory.save_context(inputs={'human': '你好，我是Tony老师'}, outputs={'ai': '不高兴认识你'})
memory.save_context(inputs={'input': '帮我回答一下1+2*3=？'}, outputs={'output': 'so easy等于7'})

print(memory.load_memory_variables({}))

print('\n')
print('------')
print(memory.chat_memory.messages)


{'history': 'Human: 你好，我是Tony老师\nAI: 不高兴认识你\nHuman: 帮我回答一下1+2*3=？\nAI: so easy等于7'}


------
[HumanMessage(content='你好，我是Tony老师', additional_kwargs={}, response_metadata={}), AIMessage(content='不高兴认识你', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='帮我回答一下1+2*3=？', additional_kwargs={}, response_metadata={}), AIMessage(content='so easy等于7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


## 使用LangGraph

In [6]:
# 举例二：推荐使用LangGraph（也可以使用上面的自动注入）
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph
from langgraph.graph.message import MessagesState  # 正确的导入
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
import os
import dotenv

# 1. 定义节点：调用 LLM 的函数
dotenv.load_dotenv(dotenv_path='../.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")

model = ChatOpenAI(
    model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)


def call_model(state: MessagesState):
    # state["messages"] 是历史消息列表
    response = model.invoke(state["messages"])
    return {"messages": [response]}  # 必须返回字典，包含新增的消息


# 2. 构建图
# InMemorySaver 会将历史消息自动保存在内存中，通过 thread_id 区分不同会话。
memory = InMemorySaver()
builder = StateGraph(MessagesState)

# 添加节点
builder.add_node("chatbot", call_model)

# 添加边：入口点 -> chatbot -> 结束
builder.set_entry_point("chatbot")
builder.set_finish_point("chatbot")

# 编译图，传入 checkpointer
graph = builder.compile(checkpointer=memory)

# 3. 调用
config = {"configurable": {"thread_id": "1"}}
# 注意：消息必须是 BaseMessage 对象列表
input_message = HumanMessage(content="你好,我叫迪迦,我喜欢吃棒棒糖")
response = graph.invoke({"messages": [input_message]}, config=config)

# 输出 AI 回复
print(response["messages"][-1].content)

# 第二次调用（自动记住历史）
input_message2 = HumanMessage(content="我叫什么名字？回答名字就行")
response2 = graph.invoke({"messages": [input_message2]}, config=config)
# 输出 AI 回复
print(response2["messages"][-1].content)
# 第三次调用（自动记住历史）
input_message3 = HumanMessage(content='我喜欢吃什么？')
response3 = graph.invoke({"messages": [input_message3]}, config=config)

print(response3["messages"][-1].content)

你好，迪迦！很高兴认识你。棒棒糖确实很美味，甜甜的，五颜六色的，让人看了心情都会变好。你最喜欢什么口味的棒棒糖呢？
迪迦
棒棒糖


## ConversationChain(方法已经过期了，不在维护)依然使用上面的自动注入

## ConversationBufferWindowMemory(方法已经过期了，不在维护)依然使用上面的自动注入

In [27]:
# 举例：结合llm、chain的使用
from langchain_classic.memory.buffer_window import ConversationBufferWindowMemory
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv(dotenv_path='../.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
# 自定义提示词模版
prompt_template = PromptTemplate.from_template(
    "以下是人类与AI之间的对话，AI表现的很健谈，并提供了大量来自上下文的具体细节,如果AI不知道问题的答案，它会表示不知道，当前对话{history}Human:{question}AI:''"
)
# 创建大模型
llm = ChatOpenAI(
    model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)
# 示例好，设置窗口阀值
memory = ConversationBufferWindowMemory(k=1)

# 定义LLMChain
conversation_with_summary = LLMChain(
    llm=llm,
    prompt=prompt_template,
    memory=memory,
    # 日志记录
    # verbose=True
)
# 执行链
res1 = conversation_with_summary.invoke({"question": "你好，我的名字是孙悟空"})

res2 = conversation_with_summary.invoke({"question": "我还有三个徒弟"})

res3 = conversation_with_summary.invoke({"question": "我今年100岁了"})

res4 = conversation_with_summary.invoke({"question": "我叫什么名字?"})

print(res4)



{'question': '我叫什么名字?', 'history': 'Human: 我今年100岁了\nAI: 哇，100岁了！这真是一个了不起的年龄，充满了智慧和经验。在这一百年里，你一定见证了许多历史性的时刻，经历了无数的变迁。能否分享一下你这一生中印象最深刻的一段经历？或者，是什么让你保持了如此长久的活力和好奇心呢？', 'text': '我目前无法确定您的名字，因为对话中没有提供相关信息。如果您愿意，可以告诉我您的名字，或者分享一些关于您自己的信息，我会尽力记住并称呼您。'}


In [4]:
# 现在的替代方案：trim_messages

'''
你可以根据 Token 数量 而不是仅仅根据 消息条数 来进行修剪，且支持保留最后几条消息的同时保留系统提示词
优点：
    1、更智能的修剪：旧版的 BufferWindowMemory 只能数消息条数，完全不顾及每条消息的长度（可能 1 条消息就占了 10k Token）。trim_messages 可以精确控制 Token 总量，防止超长上下文带来的额外成本。
    2、保留 System Prompt：在旧版中，如果你不小心，WindowMemory 可能会把你的系统提示词（Persona）给删掉。现代的 trim_messages 允许你设置 include_system=True，确保核心人设永远不会丢失。
    3、调试性：这是 LCEL 的一部分，你可以随时在日志里看到修剪前后的消息列表，不再是藏在 Memory 对象内部的黑盒。
'''
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages.utils import count_tokens_approximately
import os
import dotenv

dotenv.load_dotenv(dotenv_path='../.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")

model = ChatOpenAI(
    model='tongyi-xiaomi-analysis-pro',
    temperature=0.6
)

# 1. 定义修剪器 (取代 WindowMemory)
# 这会保持最近的 10 条消息，如果超过 Token 限制则会自动切除
trimmer = trim_messages(
    max_tokens=20000,
    strategy="last",
    # token_counter=model，即让模型自身去计算。
    # 有些模型LangChain 还没为这个模型适配 Token 计数功能。
    token_counter=count_tokens_approximately,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

# 2. 定义 Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个非常友好的助手。"),
    MessagesPlaceholder(variable_name="history"),  # 这里会动态填入历史
    ("human", "{input}"),
])

# 3. 组合链 (将修剪逻辑作为链的一环)
chain = (
        RunnablePassthrough.assign(
            history=lambda x: trimmer.invoke(x["history"])
        )
        | prompt
        | model
)
# 配合 RunnableWithMessageHistory 使用即可
store = {}


def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)
# 调用
config = {"configurable": {"session_id": "session1"}}
response = chain_with_history.invoke({"input": "你好，我叫小明"}, config=config)
response1 = chain_with_history.invoke({"input": "我叫什么名字？"}, config=config)
print(response1.content)

你叫小明。


### InMemoryChatMessageHistory

In [5]:
# InMemoryChatMessageHistory 是 LangChain 中的一个内存型消息历史记录器，用于在对话过程中临时存储 AI 和用户之间的消息记录。
from langchain_core.chat_history import InMemoryChatMessageHistory
from loguru import logger
from llm.my_llm import model

# 创建内存聊天历史记录实例，用于存储对话消息
history = InMemoryChatMessageHistory()

# 添加用户消息到聊天历史记录
history.add_user_message("我叫豆包，我的爱好是整蛊😜")

# print(history.messages)
# 调用语言模型处理聊天历史中的消息
ai_message = model.invoke(history.messages)

# 记录并输出AI回复的内容
logger.info(f"第一次回答AI回复：\n{ai_message.content}")

# 将AI回复添加到聊天历史记录中
history.add_ai_message(ai_message)

# 添加新的用户消息到聊天历史记录
history.add_user_message("我叫什么？我的爱好是什么")

# 再次调用语言模型处理更新后的聊天历史
ai_message2 = model.invoke(history.messages)

# 记录并输出第二次AI回复的内容
logger.info(f'第二次回答回复：\n{ai_message2.content}')

# 将第二次AI回复添加到聊天历史记录中
history.add_ai_message(ai_message2)

# 遍历并输出所有聊天历史记录中的消息内容
# for message in history.messages:
#     logger.info(message.content)


2026-06-06 18:38:50.778 | INFO     | __main__:<module>:18 - 第一次回答AI回复：
哇！豆包你好！😄 整蛊这个爱好也太有才了吧！看来你是个充满创意又喜欢制造小惊喜（或小惊吓）的开心果！🎭

为了帮你更好地“施展才华”，这里有一些**安全、有趣、不伤感情**的整蛊小建议，请根据对象和场合谨慎使用哦！👇

---

### **经典入门级整蛊（适合朋友/熟人）**
1. **“隐形墨水”便签**  
   ✅ **方法**：用柠檬汁在纸条上写秘密，朋友收到后用吹风机加热就能看到内容。  
   ✅ **效果**：朋友一脸懵，以为你写了“天书”，加热后突然揭晓，笑翻全场！🔥

2. **“自动喷水”纸巾盒**  
   ✅ **方法**：在纸巾盒下方钻小孔，连接到装满水的小气球，抽纸时水会喷出（提前告知对方是玩笑）。  
   ✅ **效果**：湿漉漉的纸巾+惊吓表情，瞬间破防！💦

3. **“薯片大盗”陷阱**  
   ✅ **方法**：把薯片袋底部剪开，塞入另一袋薯片，封好开口。朋友以为吃到最后一口，却突然发现“底部有惊喜”！  
   ✅ **效果**：吃货的崩溃现场，绝对意想不到！🍿

---

### **进阶脑洞级整蛊（适合关系铁的损友）**
1. **“悬浮手机”幻象**  
   ✅ **方法**：用细线绑住手机，藏在口袋里，假装手机突然飘在空中！配合“哇！它成精了！”的惊呼。  
   ✅ **效果**：科技感整蛊，对方会怀疑自己的眼睛！📱✨

2. **“自动打嗝”饮料杯**  
   ✅ **方法**：在杯底放一块小苏打，倒醋后迅速盖上盖子（留小孔）。打开时，杯盖会“砰”地弹飞，像打嗝一样！  
   ✅ **效果**：物理整蛊+化学反应，双重惊喜！🧪

3. **“会说话的抱枕”**  
   ✅ **方法**：在抱枕里藏录音笔，循环播放“别摸我！”“痒痒！”等魔性语音。朋友一抱就“诈尸”！  
   ✅ **效果**：深夜恐怖片既视感，适合胆子大的朋友！👻

---

### **⚠️ 整蛊必看安全守则！**
1. **对象要选对**：避免对内向、敏感、或正在经历压力的人整蛊！  
2. **分寸是关键**：绝不涉及人身安全、隐私、或让人难堪的内容（比如公开场合出糗）。  
3. **

### LCEL调用:RunnableWithMessageHistory

In [9]:
# 通过 RunnableWithMessageHistory 我们可以把任意 Runnable包装起来，并结合 InMemoryChatMessageHistory 来实现多轮对话。
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory, RunnableConfig
from llm.my_llm import model
from loguru import logger

# 自定义prompt
prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="history"),  # 用于插入历史消息
    ("human", "{input}")
])

parser = StrOutputParser()
# 构建处理链：将提示词模板、语言模型和输出解析器组合
chain = prompt | model | parser

# 创建内存聊天历史记录实例，用于存储对话历史
history = InMemoryChatMessageHistory()

# 创建带消息历史的可运行对象，用于处理带历史记录的对话
runnable = RunnableWithMessageHistory(
    chain,
    get_session_history=lambda session_id: history,
    input_messages_key="input",  # 指定输入建
    history_messages_key="history"  # 指定历史消息键
)
# 清空历史记录
history.clear()

# 配置运行时参数，设置会话ID
config = RunnableConfig(configurable={"session_id": "default"})
logger.info(runnable.invoke({"input": "我叫豆包，我的爱好是整蛊😜"}, config))
print('---------------')
logger.info(runnable.invoke({"input": "我叫什么，我的爱好是什么？"}, config))



2026-06-06 18:54:06.462 | INFO     | __main__:<module>:34 - 哈哈，豆包你好！看来你是个喜欢制造惊喜（和惊吓）的整蛊小能手呢！😄 作为你的“整蛊顾问”，我来帮你升级一下整蛊技能库，同时确保你的整蛊既有趣又安全无害：

---

### **🎭 安全第一！整蛊黄金法则**
1. **绝对禁区**：  
   ❌ 不涉及人身安全（如绊倒、尖锐物品）  
   ❌ 不侵犯隐私（如偷看手机、翻私人物品）  
   ❌ 不伤害自尊（如公开场合羞辱）  
   ❌ 不针对心理脆弱人群  
   **核心原则：让对方事后能笑着骂你“坏家伙”！**

---

### **💡 高能整蛊创意库（附安全操作指南）**
#### **🏠 家居整蛊**
- **“会跳舞的零食袋”**  
  在薯片袋里塞一个震动按摩头，封好口。当朋友打开时，零食袋突然“活”了！  
  ✅ 提示：提前确认对方没有心脏问题，震动强度调至“痒痒”级别。

- **“自动喷水抱枕”**  
  在抱枕里藏一个小型喷水装置，连接到坐垫压力传感器。朋友一坐下……惊喜水花四溅！  
  ✅ 提示：提前告知对方“今天有整蛊活动”，避免误伤贵重物品。

#### **🍔 餐桌整蛊**
- **“反转酱料瓶”**  
  将番茄酱瓶盖内侧贴上保鲜膜，再拧紧。朋友用力挤时，酱料会从瓶底喷出！  
  ✅ 提示：酱料量别太多，避免弄脏衣服。

- **“隐形可乐罐”**  
  可乐罐底部开小孔，用胶带封住。朋友喝到一半，空气进入，可乐会从底部漏出！  
  ✅ 提示：提前准备纸巾，营造“帮你擦手”的贴心假象。

#### **💻 办公/学习整蛊**
- **“无限弹窗电脑壁纸”**  
  把同事电脑桌面换成会自动弹出“恭喜中奖！”的动态网页，并隐藏快捷键。  
  ✅ 提示：5分钟后主动“帮忙”解决，并送上零食补偿。

- **“自动缩小的便利贴”**  
  在同事键盘缝隙贴满便利贴，上面写着“你被整蛊啦！”——每撕一张，新贴纸就弹出！  
  ✅ 提示：贴纸用可撕材质，不留胶痕。

---

### **🚨 整蛊失败急救指南**
如果对方真的生气了：  
1. **立刻道歉**：“对不起！我没想到你会介意，下次一定注意！”  
2. **补偿行动**：请喝奶茶/

---------------


2026-06-06 18:54:11.680 | INFO     | __main__:<module>:36 - 根据我们之前的对话，你的信息是这样的：  

👤 **你的名字**：豆包  
🎭 **你的爱好**：整蛊（喜欢制造有趣又无害的惊喜/恶作剧！）  

如果你是在测试我的记忆力——那放心，**我记住了**！😄  
如果你是想开启新一轮整蛊计划——随时告诉我，我随时待命当你的“军师”！💡  
（悄悄说：下次整蛊成功记得来分享战报，我会帮你写个“整蛊英雄事迹”版本！）


### 记忆窗口裁剪

In [11]:
# 记忆裁剪是指在长时间对话中，有选择地保留、压缩或丢弃部分历史消息，以保证模型的推理性能和成本可控。trim_messages 是 LangChain 中提供的一个工具函数，用于从消息列表中裁剪出“最近 N 条”消息。

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableConfig
from llm.my_llm import model

# 创建提示模版
prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder('history'),
    ('human', '{question}')
])

# 存储会话历史
store = {}

# 保留的历史轮数
k = 2


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """获取或创建会话历史"""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    # 自动修剪：只保留最近 k 轮对话（2k 条消息）
    history = store[session_id]
    if len(history.messages) > k * 2:
        # 保留最近的消息
        messages_to_keep = history.messages[-k * 2:]
        history.clear()
        history.add_messages(messages_to_keep)

    return history


# 创建带历史的链
chain = RunnableWithMessageHistory(
    prompt | model,
    get_session_history,
    input_messages_key='question',
    history_messages_key='history'
)

# 配置
config = RunnableConfig(configurable={'session_id': 'demo'})

# 主循环
print("开始对话（输入 'quit' 退出）")

while True:
    question = input("\n输入问题：")
    if question.lower() in ['quit', 'exit', 'q']:
        break
    response = chain.invoke({'question': question}, config)
    print("AI回答：", response.content)

    # 可选：显示当前历史信息数
    history = get_session_history("demo")
    print(f'[当前历史信息数：{len(history.messages)}]')






开始对话（输入 'quit' 退出）
AI回答： 是的，我是Claude，由Anthropic开发的AI助手。我的目标是提供有帮助、诚实且无害的信息和对话。

有什么我可以帮助你的问题或任务吗？无论是回答问题、提供信息、讨论话题，还是协助完成写作、分析等任务，我都很乐意为你提供支持。
[当前历史信息数：2]
AI回答： 是的，我可以写代码！我能够使用多种编程语言，包括：

- Python
- JavaScript/TypeScript  
- Java
- C/C++
- C#
- Go
- Rust
- PHP
- Ruby
- SQL
- HTML/CSS

我可以帮助你：
- 编写新代码
- 调试和修复代码问题
- 解释代码功能
- 优化代码性能
- 提供编程建议和最佳实践

你想用什么编程语言，或者有什么具体的编程任务需要帮助吗？
[当前历史信息数：4]
AI回答： 我的名字是Claude。
[当前历史信息数：4]
AI回答： Hi! 👋  
我是Claude，很高兴见到你！有什么我能帮你的吗？无论是编程问题、日常对话，还是其他任何话题，随时告诉我~ 😊
[当前历史信息数：4]
AI回答： Hello! 👋  
很高兴再次见到你！我是Claude，随时准备和你聊天或帮忙解决问题。今天想聊点什么？ 😄
[当前历史信息数：4]
AI回答： 您提到的问题可能没有在当前对话中同步过来哦 😊  
如果您方便的话，可以重新描述一下具体的问题内容，我会立刻为您解答！  
无论是技术、学习、生活建议还是其他任何需求，随时告诉我~ 🌟
[当前历史信息数：4]


### Redis存储

In [3]:
from langchain_community.chat_message_histories import RedisChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableConfig
from llm.my_llm import model
from loguru import logger

# Redis配置
REDIS_URL = "redis://localhost:6379/0"

# 创建提示模版
prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder("history"),
    ("human", "{question}")
])
# 保留的历史轮数
k = 2


def get_session_history(session_id: str) -> RedisChatMessageHistory:
    """获取或创建会话历史（使用 Redis）"""
    # 创建Redis历史对象
    history = RedisChatMessageHistory(
        session_id=session_id,
        url=REDIS_URL,
        ttl=3600
    )

    # 自动修剪：只保留最近k轮对话（2k条消息）
    if len(history.messages) > k * 2:
        # 保留最近的消息
        messages_to_keep = history.messages[-k * 2:]
        history.clear()
        history.add_messages(messages_to_keep)

    return history


#  创建带历史的链
chain = RunnableWithMessageHistory(
    prompt | model,
    get_session_history,
    input_messages_key='question',
    history_messages_key='history'
)

# 配置
config = RunnableConfig(configurable={"session_id": "demo"})

# 主循环
print("开始对话（输入'quit'退出）")

while True:
    question = input("\n输入问题")
    if question.lower() in ['quit', 'exit', 'q']:
        break
    response = chain.invoke({"question": question}, config)
    logger.info(f"AI回答：{response.content}")

    # 可选：显示当前历史消息数
    history = get_session_history('demo')
    logger.info(f'[当前历史消息数：{len(history.messages)}]')










开始对话（输入'quit'退出）


2026-06-09 19:55:23.900 | INFO     | __main__:<module>:56 - AI回答：Hello! How can I help you today?
2026-06-09 19:55:23.905 | INFO     | __main__:<module>:60 - [当前历史消息数：2]
2026-06-09 19:55:43.814 | INFO     | __main__:<module>:56 - AI回答：哇，太棒了！多才多艺呀。🎤💃 

那你平时最喜欢唱什么类型的歌，跳什么风格的舞蹈呢？有没有什么特别拿手的曲目或舞蹈？
2026-06-09 19:55:43.820 | INFO     | __main__:<module>:60 - [当前历史消息数：4]
2026-06-09 19:56:29.449 | INFO     | __main__:<module>:56 - AI回答：虽然我没有身体，不能像你一样在舞台上真正地“唱跳”，但我可是个“脑力”小能手哦！我会的东西还挺多的，主要可以帮你做这些：

1. **创意与写作**：帮你写歌词、编故事、写诗、写文章。如果你要准备新的唱跳作品，我可以帮你找灵感、写Rap词或者策划舞台概念！
2. **知识与解答**：无论是历史、科学、文化还是生活常识，我都能为你解答，相当于一个随身携带的百科全书。
3. **语言与翻译**：精通多国语言，可以帮你翻译、润色外语，或者陪你练习外语对话。
4. **逻辑与编程**：帮你写代码、找Bug、做数学题、进行逻辑推理和数据分析。
5. **休闲与陪伴**：陪你聊天解闷、玩文字游戏、做旅游攻略，或者给你推荐好听的歌单和舞蹈风格。

简单来说，只要你能想到的文字类需求，我基本都能帮忙。

你想先让我展示点什么？或者，需要我帮你写一段酷炫的开场Rap词吗？😎🎤
2026-06-09 19:56:29.458 | INFO     | __main__:<module>:60 - [当前历史消息数：4]
2026-06-09 20:01:08.693 | INFO     | __main__:<module>:56 - AI回答：既然你会唱跳，那我会给你**写Rap词、排舞蹈队形，顺便再给你递个篮球**！🏀🎤

说点实际的

In [1]:
from langchain_core.runnables import RunnableConfig
from llm.my_llm import model
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import RedisChatMessageHistory
import gradio as gr

# 定义不同角色的系统提示语
ROLES = {
    "通用助手": "你是无所不知的 AI 助手。",
    "段子手": "你是脱口秀演员，回答必须带 1 个梗。",
    "英语老师": "你是耐心英语老师，先用英文回答，再给中文翻译。",
    "代码审查员": "你是严格的代码审查员，指出代码问题并给出改进建议。",
}




def get_session_history(session_id: str) -> RedisChatMessageHistory:
    """
    根据会话 ID 获取 Redis 中的消息历史记录。

    参数:
        session_id (str): 会话唯一标识符。

    返回:
        RedisChatMessageHistory: 与该会话关联的聊天历史对象。
    """
    return RedisChatMessageHistory(
        session_id=session_id,
        url='redis://localhost:6379/0',
        key_prefix="chat:",
        ttl=None
    )


def build_chain(role: str):
    """
    构建一个包含系统提示和用户输入的处理链。

    参数:
        role (str): 当前使用的角色名称。

    返回:
        Chain: 包含提示模板和语言模型的可执行链。
    """
    system = ROLES[role]
    prompt = ChatPromptTemplate.from_messages([
        ("system", system),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}")
    ])

    return prompt | model


def chat_fn(message, history, role):
    """
    处理用户的聊天消息，并流式返回响应结果。

    参数:
        message (str): 用户发送的消息内容。
        history (list): 当前对话的历史记录。
        role (str): 当前使用的角色名称。

    生成:
        tuple: 更新后的聊天记录和清空输入框的内容。
    """
    chain_with_history = RunnableWithMessageHistory(
        build_chain(role),
        get_session_history,
        input_messages_key="question",
        history_messages_key="history"
    )
    partial = ""
    config = RunnableConfig(configurable={"session_id": role})
    for chunk in chain_with_history.stream({"question": message}, config):
        partial += chunk.content
        yield history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": partial}
        ], ""


def switch_role(new_role):
    """
    切换当前角色，并更新显示信息及清空聊天记录。

    参数:
        new_role (str): 新的角色名称。

    返回:
        tuple: 更新后的角色显示文本、清空聊天记录和新的角色状态。
    """
    return f"**当前角色：{new_role}**", [], new_role


# 使用 Gradio 构建 Web 界面
with gr.Blocks(title="多角色聊天") as demo:
    # 初始化当前角色状态为“通用助手”
    current_role_state = gr.State("通用助手")

    # 页面布局：左侧角色选择区，右侧聊天区域
    with gr.Row():
        # 创建角色选择界面列
        # 该代码块负责构建角色选择的UI界面，包括角色标题显示、当前角色状态显示和角色选择按钮
        with gr.Column(scale=1):
            gr.Markdown("### 选择角色")
            current_role_display = gr.Markdown("**当前角色：通用助手**")
            role_buttons = [gr.Button(role, variant="secondary") for role in ROLES.keys()]

        # 创建聊天界面的主区域布局
        # 该区域包含聊天显示区、消息输入框和发送按钮
        with gr.Column(scale=4, elem_classes=["chat-area"]):
            # 聊天机器人组件，用于显示对话历史
            chatbot = gr.Chatbot(label="聊天区", height='70vh')
            # 文本输入框组件，用于用户输入消息
            msg = gr.Textbox(label="输入你的消息", placeholder="请输入...", scale=10)

            # 发送按钮组件，用于提交用户输入的消息
            send_btn = gr.Button("发送", variant="primary")

    # 绑定发送按钮点击事件
    send_btn.click(
        fn=chat_fn,
        inputs=[msg, chatbot, current_role_state],
        outputs=[chatbot, msg]
    )

    # 绑定每个角色按钮的点击事件
    for btn in role_buttons:
        btn.click(
            fn=lambda r=btn.value: switch_role(r),
            inputs=None,
            outputs=[current_role_display, chatbot, current_role_state]
        )

# 启动 Gradio 应用
if __name__ == "__main__":
    demo.launch()


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
